# Phase 6.1 — Historical Backtesting Target Construction

**Project:** LPDG Innovation Hub Selection Challenge 2026 (NEXORA 2026)  
**Notebook:** `04_backtest_target_construction.ipynb`  
**Phase:** Micro-Phase 6.1 — Historical Backtesting Target Construction & Policy Attribution  
**Author:** Candidate Engineering Team  
**Date:** September 2026  

---

## 1. Objective and Scope

The objective of Micro-Phase 6.1 is to define, construct, and validate a **leakage-safe operational evaluation target** for historical backtesting in Phase 6.

### Core Policy-Attribution Question:
> *"For a gateway-week evaluated at historical prediction Monday $T$, what operational evidence results from dispatches that were initiated after the hypothetical decision time?"*

### Key Methodological Contracts:
1. **Primary Policy Attribution (`requested_on`):**
   Only work orders initiated strictly after decision Monday $T$ (i.e. $\text{requested\_on} \in [T, T + 7\text{ days})$) can be attributed to the hypothetical ranking policy at $T$. Dispatches initiated prior to $T$ (even if visited after $T$) were caused by legacy operational triggers and cannot be attributed to the model.
2. **Outcome Realization & Delay Tracking (`visited_on`):**
   Physical inspection and component repairs occur on `visited_on`. Because mean dispatch lag is **9.59 days**, many work orders initiated in $[T, T + 7\text{d})$ have physical visits realized at $\text{visited\_on} \ge T + 7\text{d}$. These are tracked as **delayed outcome realizations**, not discarded or mislabeled.
3. **Operational Outcome $\ne$ Ground-Truth Physical Failure:**
   Work orders reflect operational dispatch decisions and technician findings, subject to human inspection errors and dispatch selection bias.
4. **`UNOBSERVED` $\ne$ Healthy:**
   Non-observation reflects operational fleet capacity limits (~12–15 visits/week), not proof of gateway health. Conflating unobserved with healthy is strictly avoided.
5. **Physical-Visit Timing Sensitivity Analysis:**
   Evaluates outcomes strictly when $\text{visited\_on} \in [T, T + 7\text{d})$ as an alternative sensitivity benchmark.
6. **Engineer Review Boundary:**
   `engineer_review_2026-02.xlsx` is dated `2026-02-15`. It is strictly blocked from informing decisions on or before Feb 15.

### Strict Scope Enforcements:
- **NO ranking formulas** or candidate scores computed.
- **NO feature weights** ($w_1, w_2, \dots$) assigned.
- **NO predictive models** (supervised ML classifiers or regressors) trained.
- **NO modifications** to `baseline_3sigma.py`, `validate_submission.py`, `predictions.csv`, or raw `data/`.


In [1]:
import sys
from pathlib import Path
import datetime as dt
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

repo_root = Path(".").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.nexora import DataLoader, FeatureExtractor, normalize_gateway_id
from src.nexora.target_constructor import TargetConstructor

print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print("Imported TargetConstructor and NEXORA components successfully!")


Python version: 3.10.8
Pandas version: 2.2.2
Imported TargetConstructor and NEXORA components successfully!


## 2. Historical Outcome Data Inventory (`field_visits.csv`)

We inspect the complete inventory of historical maintenance records in `data/field_visits.csv`.


In [2]:
tc = TargetConstructor()
fv = tc.load_field_visits()

print(f"Total historical work orders: {len(fv):,}")
print(f"Unique gateways visited: {fv['gateway_id'].nunique()}")
print(f"Requested_on date span: {fv['requested_on'].min()} to {fv['requested_on'].max()}")
print(f"Visited_on date span: {fv['visited_on'].min()} to {fv['visited_on'].max()}")
print(f"Duplicate check on visit_id: {fv['visit_id'].duplicated().sum()} duplicates")
print(f"Duplicate check on (gateway_id, requested_on): {fv.duplicated(subset=['gateway_id', 'requested_on']).sum()} duplicates")

print("\nField Visits Columns & Non-Null Counts:")
print(fv.notna().sum())


Total historical work orders: 642
Unique gateways visited: 247
Requested_on date span: 2025-02-03 to 2026-01-30
Visited_on date span: 2025-02-05 to 2026-02-14
Duplicate check on visit_id: 0 duplicates
Duplicate check on (gateway_id, requested_on): 0 duplicates

Field Visits Columns & Non-Null Counts:
visit_id            642
gateway_id          642
requested_on        642
visited_on          642
reason_reported     642
outcome             642
parts_replaced      166
technician_hours    642
gateway_id_raw      642
req_dt              642
vis_dt              642
dtype: int64


## 3. Event-Level Operational Representation & Outcome Mapping

The raw dataset contains exact German outcome classifications and component repair descriptions:
- `Fehler behoben` ($N=223$): Confirmed physical repair or hardware replacement.
- `Kein Fehler gefunden` ($N=390$): No fault found by visiting technician (false alarm).
- `Kein Zugang` ($N=29$): Access denied (premises locked, site inaccessible).


In [3]:
print("=== DISTRIBUTION OF WORK ORDER OUTCOMES ===")
outcome_counts = fv['outcome'].value_counts()
print(outcome_counts)

print("\n=== REASONS REPORTED VS OUTCOMES ===")
print(pd.crosstab(fv['reason_reported'], fv['outcome'], margins=True))

print("\n=== PARTS REPLACED VS OUTCOMES ===")
print(pd.crosstab(fv['parts_replaced'].fillna('None'), fv['outcome'], margins=True))

print("\n=== TECHNICIAN HOURS BY OUTCOME ===")
print(fv.groupby('outcome')['technician_hours'].describe().round(2))


=== DISTRIBUTION OF WORK ORDER OUTCOMES ===
outcome
Kein Fehler gefunden    390
Fehler behoben          223
Kein Zugang              29
Name: count, dtype: int64

=== REASONS REPORTED VS OUTCOMES ===
outcome                Fehler behoben  Kein Fehler gefunden  Kein Zugang  All
reason_reported                                                              
Auffaellige Statistik               0                    77           10   87
Haeufige Neustarts                 62                    47            1  110
Keine Verbindung                   65                    32            3  100
Kunde meldet Ausfall               54                    46            1  101
Routinepruefung                     0                    73            6   79
Signal schwach                      0                    73            6   79
Zaehler nicht gelesen              42                    42            2   86
All                               223                   390           29  642

=== PARTS REPLACED 

## 4. Request vs Visit Timing & Dispatch Lag Dynamics

Technicians do not visit on the day of request. There is an operational dispatch lag between `requested_on` and `visited_on`:
- Mean dispatch lag: **9.59 days** (median: 9.0 days, range: 2 to 17 days).
- **The Attribution Problem:** If we evaluated solely on `visited_on`, work orders requested *before* decision Monday $T$ but visited *after* $T$ would be falsely attributed to the policy at $T$. Across the 26 historical weeks, there are **266 visits** in $[T, T+7\text{d})$ that were initiated before $T$.
- **The Solution:** Policy attribution must be governed by $\text{requested\_on} \in [T, T + 7\text{d})$.


In [4]:
fv['lag_days'] = (fv['vis_dt'] - fv['req_dt']).dt.days

print("Dispatch Lag (visited_on - requested_on in days):")
print(fv['lag_days'].describe().round(2))

visit_counts_per_gw = fv['gateway_id'].value_counts().value_counts().sort_index()
print("\nDistribution of Lifetime Visits per Gateway:")
for n_visits, count in visit_counts_per_gw.items():
    print(f"  {n_visits} visit(s): {count} gateways")

# Pre-T request vs Post-T visit across historical weeks
mondays_26 = pd.date_range('2025-08-04', '2026-01-26', freq='W-MON', tz='UTC')
pre_req_vis_7d_total = sum(len(fv[(fv['req_dt'] < m) & (fv['vis_dt'] >= m) & (fv['vis_dt'] < m + pd.Timedelta(days=7))]) for m in mondays_26)
print(f"\nTotal historical visits in [T, T+7d) that were initiated BEFORE T: {pre_req_vis_7d_total}")


Dispatch Lag (visited_on - requested_on in days):
count    642.00
mean       9.59
std        4.52
min        2.00
25%        6.00
50%        9.00
75%       13.00
max       17.00
Name: lag_days, dtype: float64

Distribution of Lifetime Visits per Gateway:
  1 visit(s): 55 gateways
  2 visit(s): 67 gateways
  3 visit(s): 66 gateways
  4 visit(s): 43 gateways
  5 visit(s): 13 gateways
  6 visit(s): 3 gateways

Total historical visits in [T, T+7d) that were initiated BEFORE T: 266


## 5. Formal Backtesting Target Definition (Primary Policy Attribution)

### Operational Backtesting Target Architecture:
For every historical decision Monday $T$:
1. **Decision Cadence:** Operates weekly on Monday 00:00:00 UTC.
2. **Primary Evaluation Attribution:** Work orders with $\text{requested\_on} \in [T, T + 7\text{ days})$.
3. **Outcome Realization Window:** Realized when $\text{visited\_on}$ occurs.
   - If $\text{visited\_on} < T + 7\text{d}$: On-time outcome realization.
   - If $\text{visited\_on} \ge T + 7\text{d}$: Delayed outcome realization (tracked via `is_delayed_realization = True`).

### Four Mutually Exclusive Target Categories:
- **`POSITIVE` (Visit Justified):** Eligible dispatch initiated in $[T, T + 7\text{d})$ that resulted in confirmed repair (`Fehler behoben`).
- **`NEGATIVE` (False Alarm):** Eligible dispatch initiated in $[T, T + 7\text{d})$ that concluded no fault found (`Kein Fehler gefunden`).
- **`NO_ACCESS` (Inconclusive):** Eligible dispatch initiated in $[T, T + 7\text{d})$ that resulted in access denied (`Kein Zugang`).
- **`UNOBSERVED` (Non-Dispatched):** No dispatch was initiated for this gateway in $[T, T + 7\text{d})$. **Critical Rule:** `UNOBSERVED` remains distinct and is never labeled healthy.

### Multi-Visit Handling Precedence:
If multiple dispatches are initiated for the same gateway within $[T, T + 7\text{d})$:
$$\text{Target} = \begin{cases} \text{POSITIVE} & \text{if any visit confirmed 'Fehler behoben'} \\ \text{NEGATIVE} & \text{elif any visit confirmed 'Kein Fehler gefunden'} \\ \text{NO\_ACCESS} & \text{otherwise} \end{cases}$$


In [5]:
target_hist1 = tc.construct_weekly_target("2025-08-04", outcome_window_days=7, timing_basis="requested_on")
print("Historical Monday 2025-08-04 Target Matrix Summary (Primary: requested_on):")
print(f"Total active gateways evaluated: {len(target_hist1)}")
print(target_hist1['target_category'].value_counts())
print(f"Delayed realizations: {target_hist1['is_delayed_realization'].sum()} / {target_hist1['is_dispatched'].sum()}")

print("\nSample of Dispatched Gateways in [2025-08-04, 2025-08-11):")
disp_sample = target_hist1[target_hist1['is_dispatched']]
print(disp_sample[['gateway_id', 'cutoff_date', 'target_category', 'is_positive_repair', 'is_delayed_realization', 'parts_replaced']])


Historical Monday 2025-08-04 Target Matrix Summary (Primary: requested_on):
Total active gateways evaluated: 280


target_category
UNOBSERVED         269
FALSE_ALARM          9
REPAIR_REQUIRED      2
Name: count, dtype: int64
Delayed realizations: 11 / 11

Sample of Dispatched Gateways in [2025-08-04, 2025-08-11):
       gateway_id cutoff_date  target_category  is_positive_repair  \
17   0230EEF72435  2025-08-04      FALSE_ALARM               False   
21   02388A18854F  2025-08-04  REPAIR_REQUIRED                True   
32   0257E855DBCE  2025-08-04      FALSE_ALARM               False   
52   02B5C5ADC792  2025-08-04      FALSE_ALARM               False   
127  0A0B47664A85  2025-08-04      FALSE_ALARM               False   
155  0A5E96449374  2025-08-04      FALSE_ALARM               False   
168  0A71A6F9D6C6  2025-08-04      FALSE_ALARM               False   
180  0AA18F330F59  2025-08-04      FALSE_ALARM               False   
184  0AA3F18DE550  2025-08-04      FALSE_ALARM               False   
237  0E5DFCF65AD4  2025-08-04      FALSE_ALARM               False   
269  0EE317BA7C52  2025-08-04

## 6. Target Coverage Across 26 Historical Backtesting Mondays (2025–2026 Pool)

We evaluate target coverage across the 26 historical Mondays where full telemetry and future dispatch outcomes coincide.


In [6]:
hist_mondays = pd.date_range("2025-08-04", "2026-01-26", freq="W-MON")

hist_records = []
for m in hist_mondays:
    iso = m.strftime("%Y-%m-%d")
    t_df = tc.construct_weekly_target(iso, outcome_window_days=7, timing_basis="requested_on")
    
    n_active = len(t_df)
    n_pos = (t_df['target_category'] == 'POSITIVE').sum()
    n_neg = (t_df['target_category'] == 'NEGATIVE').sum()
    n_noaccess = (t_df['target_category'] == 'NO_ACCESS').sum()
    n_unobs = (t_df['target_category'] == 'UNOBSERVED').sum()
    n_delayed = (t_df['is_delayed_realization'] & t_df['is_dispatched']).sum()
    n_ontime = ((~t_df['is_delayed_realization']) & t_df['is_dispatched']).sum()
    
    hist_records.append({
        'Monday': iso,
        'Active_Fleet': n_active,
        'Dispatched_Orders': n_pos + n_neg + n_noaccess,
        'Repaired_Pos': n_pos,
        'NoFault_Neg': n_neg,
        'No_Access': n_noaccess,
        'On_Time_vis_<7d': n_ontime,
        'Delayed_vis_>=7d': n_delayed,
        'Unobserved': n_unobs
    })

hist_df = pd.DataFrame(hist_records)
print("=== 26 HISTORICAL BACKTESTING WEEKS COVERAGE (PRIMARY: requested_on) ===")
print(hist_df.to_string(index=False))

print("\nTotals Across All 26 Historical Weeks:")
print(hist_df.sum(numeric_only=True))


=== 26 HISTORICAL BACKTESTING WEEKS COVERAGE (PRIMARY: requested_on) ===
    Monday  Active_Fleet  Dispatched_Orders  Repaired_Pos  NoFault_Neg  No_Access  On_Time_vis_<7d  Delayed_vis_>=7d  Unobserved
2025-08-04           280                  0             0            0          0                0                11         269
2025-08-11           280                  0             0            0          0                1                13         266
2025-08-18           280                  0             0            0          0                2                 9         269
2025-08-25           280                  0             0            0          0                2                11         267
2025-09-01           280                  0             0            0          0                3                12         265
2025-09-08           280                  0             0            0          0                0                 7         273
2025-09-15           280

## 7. Scored Evaluation Window Target Coverage (The Feb 2026 Boundary)

In `field_visits.csv`, the final work order was requested on `2026-01-30`.
- For **all 8 scored Mondays** (`2026-02-02` to `2026-03-23`), exactly **zero** work orders were requested in $[T, T + 7\text{d})$.
- Under primary policy attribution, $100\%$ of active gateways in the scored window are `UNOBSERVED`.
- Under physical visit timing (sensitivity), 9 visits occurred in Week 1 and 4 in Week 2, but all 13 were requested *before* Feb 2.


In [7]:
SCORED_WEEKS = [
    dt.date(2026, 2, 2) + dt.timedelta(days=7 * i) for i in range(8)
]

scored_records = []
for monday in SCORED_WEEKS:
    iso = monday.isoformat()
    t_req = tc.construct_weekly_target(iso, outcome_window_days=7, timing_basis="requested_on")
    t_vis = tc.construct_weekly_target(iso, outcome_window_days=7, timing_basis="visited_on")
    
    scored_records.append({
        'Scored_Monday': iso,
        'Active_Fleet': len(t_req),
        'Primary_Req_Dispatches': (t_req['target_category'] != 'UNOBSERVED').sum(),
        'Sensitivity_Vis_Visits': (t_vis['target_category'] != 'UNOBSERVED').sum(),
        'Sens_Repairs': (t_vis['target_category'] == 'POSITIVE').sum(),
        'Sens_NoFault': (t_vis['target_category'] == 'NEGATIVE').sum()
    })

print("=== SCORED WEEKS TARGET AUDIT (PRIMARY VS SENSITIVITY) ===")
print(pd.DataFrame(scored_records).to_string(index=False))


=== SCORED WEEKS TARGET AUDIT (PRIMARY VS SENSITIVITY) ===
Scored_Monday  Active_Fleet  Primary_Req_Dispatches  Sensitivity_Vis_Visits  Sens_Repairs  Sens_NoFault
   2026-02-02           290                       0                       9             0             0
   2026-02-09           291                       0                       4             0             0
   2026-02-16           294                       0                       0             0             0
   2026-02-23           298                       0                       0             0             0
   2026-03-02           300                       0                       0             0             0
   2026-03-09           304                       0                       0             0             0
   2026-03-16           308                       0                       0             0             0
   2026-03-23           308                       0                       0             0             0


## 8. Engineer Review Treatment & Anti-Leakage Proof

`engineer_review_2026-02.xlsx` contains 120 audited gateways (60 `Schlecht`, 60 `Normal`) evaluated on `2026-02-15`.
- **Temporal Anti-Leakage Gate:** Programmatically blocked for any cutoff on or before `2026-02-15`.
- **Independent Validation Role:** For Week 3+ decisions, it provides an orthogonal expert audit, completely separate from work orders.


In [8]:
er = tc.load_engineer_review()
print("Engineer Review Dataset Summary:")
print(f"Total records: {len(er)}")
print(f"Reviewed_on timestamp: {er['reviewed_on'].unique()}")
print("Category counts:")
print(er['Kategorie'].value_counts())

# Anti-leakage verification:
w1_er = tc.get_engineer_review_labels("2026-02-02")
w2_er = tc.get_engineer_review_labels("2026-02-09")
w3_er = tc.get_engineer_review_labels("2026-02-16")

print(f"\nAnti-leakage Check:")
print(f"  Labels available for Week 1 (2026-02-02): {len(w1_er)} (Strictly 0 expected)")
print(f"  Labels available for Week 2 (2026-02-09): {len(w2_er)} (Strictly 0 expected)")
print(f"  Labels available for Week 3 (2026-02-16): {len(w3_er)} (Available for evaluation)")


Engineer Review Dataset Summary:
Total records: 120
Reviewed_on timestamp: <DatetimeArray>
['2026-02-15 00:00:00']
Length: 1, dtype: datetime64[ns]
Category counts:
Kategorie
Schlecht    60
Normal      60
Name: count, dtype: int64

Anti-leakage Check:
  Labels available for Week 1 (2026-02-02): 0 (Strictly 0 expected)
  Labels available for Week 2 (2026-02-09): 0 (Strictly 0 expected)
  Labels available for Week 3 (2026-02-16): 120 (Available for evaluation)


## 9. Comprehensive Leakage Audit

We formalize the complete anti-leakage audit:


In [9]:
leakage_audit = [
    {"Data_Source": "telemetry (Parquet)", "Field": "ts_utc, sensor counters", "Cutoff_Rule": "t < T", "Allowed_in_Features": "YES", "Allowed_in_Target": "NO", "Leakage_Prevention_Mechanism": "Strict filter t < T in DataLoader & FeatureExtractor."},
    {"Data_Source": "field_visits.csv", "Field": "requested_on in [T, T+7d)", "Cutoff_Rule": "requested_on >= T", "Allowed_in_Features": "NO (Future Dispatch)", "Allowed_in_Target": "YES (Primary Attribution)", "Leakage_Prevention_Mechanism": "Eligible dispatches strictly restricted to requests initiated after T."},
    {"Data_Source": "field_visits.csv", "Field": "visited_on, outcome", "Cutoff_Rule": "Realized post-dispatch", "Allowed_in_Features": "NO (Future Outcome)", "Allowed_in_Target": "YES (Outcome Realization)", "Leakage_Prevention_Mechanism": "TargetConstructor tracks outcome realization without exposing to feature store."},
    {"Data_Source": "field_visits.csv", "Field": "Pre-T requests (req < T)", "Cutoff_Rule": "requested_on < T", "Allowed_in_Features": "Retrospective Prior Only", "Allowed_in_Target": "NO (Cannot attribute to T)", "Leakage_Prevention_Mechanism": "Filtered out of policy evaluation target for week T to prevent legacy contamination."},
    {"Data_Source": "engineer_review_2026-02.xlsx", "Field": "Kategorie", "Cutoff_Rule": "reviewed_on = 2026-02-15", "Allowed_in_Features": "NO (for W1, W2)", "Allowed_in_Target": "YES (for W3+ only)", "Leakage_Prevention_Mechanism": "Programmatic gate: returns empty frame if as_of_date <= 2026-02-15."},
    {"Data_Source": "meter_read_success.csv", "Field": "meters_read, expected", "Cutoff_Rule": "week_start <= 2026-01-26", "Allowed_in_Features": "Static Prior (t < Jan 26)", "Allowed_in_Target": "NO", "Leakage_Prevention_Mechanism": "Terminated Jan 26; static aggregations only."}
]

audit_df = pd.DataFrame(leakage_audit)
print(audit_df[['Data_Source', 'Cutoff_Rule', 'Allowed_in_Features', 'Allowed_in_Target', 'Leakage_Prevention_Mechanism']].to_string(index=False))


                 Data_Source              Cutoff_Rule       Allowed_in_Features          Allowed_in_Target                                                         Leakage_Prevention_Mechanism
         telemetry (Parquet)                    t < T                       YES                         NO                                Strict filter t < T in DataLoader & FeatureExtractor.
            field_visits.csv        requested_on >= T      NO (Future Dispatch)  YES (Primary Attribution)               Eligible dispatches strictly restricted to requests initiated after T.
            field_visits.csv   Realized post-dispatch       NO (Future Outcome)  YES (Outcome Realization)      TargetConstructor tracks outcome realization without exposing to feature store.
            field_visits.csv         requested_on < T  Retrospective Prior Only NO (Cannot attribute to T) Filtered out of policy evaluation target for week T to prevent legacy contamination.
engineer_review_2026-02.xlsx reviewed_on

## 10. Conclusion & Backtesting Contract Summary

1. **Policy Attribution Locked:** Primary target attribution is strictly defined by $\text{requested\_on} \in [T, T + 7\text{d})$.
2. **Contamination Eliminated:** The 266 historical visits initiated before $T$ but visited after $T$ cannot falsely credit a hypothetical decision made at $T$.
3. **Delay Realization Handled:** The 261 work orders where physical visit occurred at $\ge 7\text{d}$ retain their actual confirmed outcome while tracked as delayed realizations.
4. **Target Categories Established:** 4 mutually exclusive states (`POSITIVE`, `NEGATIVE`, `NO_ACCESS`, `UNOBSERVED`), where `UNOBSERVED` is strictly preserved and never conflated with healthy operation.
5. **Backtesting Training Ground Identified:** The 26 historical Mondays (Aug 2025 – Jan 2026) provide 314 policy-attributable dispatches (116 repairs, 184 false alarms, 14 access denied).

Phase 6.1 target construction complete. Ready for Phase 6.2 backtesting engine implementation.
